# 05-2. 勾配降下法 — 動かして確かめる

📖 解説: [`../02_gradient_descent.md`](../02_gradient_descent.md)

## このノートで触るもの
1. 基本の勾配降下法 + 軌跡可視化
2. 学習率の影響 (発散・遅すぎ・適切)
3. SGD vs Mini-batch GD
4. モメンタム
5. Adam 実装
6. 主要オプティマイザの比較

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_gradient_descent.md)](../02_gradient_descent.md)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatLogSlider, FloatSlider

%matplotlib inline
rng = np.random.default_rng(42)

## 1. 基本の勾配降下法 + 軌跡可視化

$$
\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \nabla L(\boldsymbol{\theta}_t)
$$

対応するコード: `params = params - lr * grad_fn(params)`

In [ ]:
def f(x):
    """目的関数: 楕円形のお椀 f(x, y) = x² + 5y²."""
    return x[0]**2 + 5 * x[1]**2

grad_f = jax.grad(f)

def gradient_descent(x0, lr, n_iter):
    x = x0
    traj = [x]
    for _ in range(n_iter):
        x = x - lr * grad_f(x)
        traj.append(x)
    return jnp.array(traj)

# 実行
x0 = jnp.array([4.0, 1.5])
traj = gradient_descent(x0, lr=0.1, n_iter=30)

# 等高線 + 軌跡
x_g = np.linspace(-5, 5, 100)
y_g = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_g, y_g)
Z = X**2 + 5 * Y**2

fig, ax = plt.subplots(figsize=(10, 6))
ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
ax.plot(traj[:, 0], traj[:, 1], 'r-o', markersize=4, label=f'軌跡 ({len(traj)-1} steps)')
ax.plot(0, 0, 'g*', markersize=20, label='最適解 (0, 0)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('勾配降下法の軌跡 ($f = x^2 + 5y^2$, $\\eta = 0.1$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 2. 学習率の影響

学習率 $\eta$ で挙動が劇的に変わる:
- 大きすぎ → 発散
- 適切 → 収束
- 小さすぎ → 遅い

In [ ]:
def show_lr_effect(lr: float) -> None:
    x0 = jnp.array([4.0, 1.5])
    try:
        traj = gradient_descent(x0, lr=lr, n_iter=30)
        if jnp.any(jnp.isnan(traj)) or jnp.any(jnp.abs(traj) > 100):
            status = '❌ 発散!'
        else:
            final_loss = float(f(traj[-1]))
            status = f'✅ 最終損失 = {final_loss:.4e}'
    except Exception as e:
        status = f'❌ エラー: {e}'
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
    if not jnp.any(jnp.isnan(traj)):
        traj_clip = jnp.clip(traj, -10, 10)
        ax.plot(traj_clip[:, 0], traj_clip[:, 1], 'r-o', markersize=3)
    ax.plot(0, 0, 'g*', markersize=20, label='最適解')
    ax.set_xlim(-10, 10); ax.set_ylim(-5, 5)
    ax.set_title(f'$\\eta = {lr}$  {status}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(show_lr_effect, lr=FloatLogSlider(value=0.1, base=10, min=-3, max=0.3, step=0.1));

## 3. SGD vs Mini-batch — 線形回帰で比較

$$
L(\boldsymbol{\theta}) = \frac{1}{N} \sum_{i=1}^N (y_i - \boldsymbol{\theta}^\top \mathbf{x}_i)^2
$$

GD: 全データで勾配 / SGD: 1サンプル / Mini-batch: 小さい集合

In [ ]:
# データ生成: y = 2x + 1 + ノイズ
n = 1000
X_data = rng.uniform(-5, 5, (n, 1))
y_data = 2 * X_data[:, 0] + 1 + rng.normal(0, 0.5, n)
X_data = np.column_stack([X_data, np.ones(n)])  # バイアス項追加

def loss(theta, X, y):
    return jnp.mean((X @ theta - y)**2)

grad_loss = jax.grad(loss)

def train(method: str, lr: float = 0.01, n_epochs: int = 30, batch_size: int = 32):
    theta = jnp.zeros(2)
    history = []
    for epoch in range(n_epochs):
        if method == 'GD':
            theta = theta - lr * grad_loss(theta, X_data, y_data)
        elif method == 'SGD':
            for _ in range(n // batch_size):  # 同じ計算量
                idx = rng.integers(0, n)
                theta = theta - lr * grad_loss(theta, X_data[idx:idx+1], y_data[idx:idx+1])
        elif method == 'Mini-batch':
            indices = rng.permutation(n)
            for start in range(0, n, batch_size):
                batch = indices[start:start+batch_size]
                theta = theta - lr * grad_loss(theta, X_data[batch], y_data[batch])
        history.append(float(loss(theta, X_data, y_data)))
    return theta, history

fig, ax = plt.subplots(figsize=(10, 5))
for method in ['GD', 'SGD', 'Mini-batch']:
    theta, hist = train(method)
    ax.plot(hist, label=f'{method}: 最終 θ = {np.array(theta).round(3).tolist()}')
ax.set_xlabel('epoch'); ax.set_ylabel('損失')
ax.set_yscale('log')
ax.set_title('GD vs SGD vs Mini-batch')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f'真値: w = 2.0, b = 1.0')

## 4. モメンタム

$$
\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla L(\boldsymbol{\theta}_t), \quad \boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \mathbf{v}_{t+1}
$$

対応するコード: `v = beta * v + grad; params = params - lr * v`

In [ ]:
def momentum_descent(x0, lr=0.05, beta=0.9, n_iter=30):
    x = x0
    v = jnp.zeros_like(x0)
    traj = [x]
    for _ in range(n_iter):
        v = beta * v + grad_f(x)
        x = x - lr * v
        traj.append(x)
    return jnp.array(traj)

# モメンタム vs プレーン GD
x0 = jnp.array([4.0, 1.5])
traj_gd = gradient_descent(x0, lr=0.05, n_iter=30)
traj_mom = momentum_descent(x0, lr=0.05, beta=0.9, n_iter=30)

fig, ax = plt.subplots(figsize=(11, 6))
ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
ax.plot(traj_gd[:, 0], traj_gd[:, 1], 'r-o', markersize=4, label=f'GD (η=0.05)')
ax.plot(traj_mom[:, 0], traj_mom[:, 1], 'b-s', markersize=4, label=f'Momentum (β=0.9)')
ax.plot(0, 0, 'g*', markersize=20, label='最適解')
ax.set_xlim(-5, 5); ax.set_ylim(-3, 3)
ax.set_title('モメンタム vs 通常 GD')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f'GD       最終: x={np.array(traj_gd[-1])},  loss={float(f(traj_gd[-1])):.6f}')
print(f'Momentum 最終: x={np.array(traj_mom[-1])},  loss={float(f(traj_mom[-1])):.6f}')

## 5. Adam の実装

$$
\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \nabla L
$$

$$
\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) (\nabla L)^2
$$

$$
\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}
$$

In [ ]:
def adam(x0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, n_iter=30):
    x = x0
    m = jnp.zeros_like(x0)
    v = jnp.zeros_like(x0)
    traj = [x]
    for t in range(1, n_iter + 1):
        g = grad_f(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x = x - lr * m_hat / (jnp.sqrt(v_hat) + eps)
        traj.append(x)
    return jnp.array(traj)

x0 = jnp.array([4.0, 1.5])
traj_adam = adam(x0, lr=0.3, n_iter=30)

fig, ax = plt.subplots(figsize=(11, 6))
ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
ax.plot(traj_adam[:, 0], traj_adam[:, 1], 'm-^', markersize=4, label=f'Adam (η=0.3)')
ax.plot(0, 0, 'g*', markersize=20, label='最適解')
ax.set_xlim(-5, 5); ax.set_ylim(-3, 3)
ax.set_title('Adam: 適応的学習率')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. 主要オプティマイザの比較

In [ ]:
x0 = jnp.array([4.0, 1.5])
n_iter = 50

results = {
    'GD (η=0.05)':       gradient_descent(x0, lr=0.05, n_iter=n_iter),
    'Momentum (β=0.9)':  momentum_descent(x0, lr=0.05, beta=0.9, n_iter=n_iter),
    'Adam (η=0.3)':      adam(x0, lr=0.3, n_iter=n_iter),
}

fig, ax = plt.subplots(figsize=(11, 5))
for name, traj in results.items():
    losses = [float(f(p)) for p in traj]
    ax.plot(losses, label=name, linewidth=2)
ax.set_xlabel('反復')
ax.set_ylabel('損失 (対数)')
ax.set_yscale('log')
ax.set_title('オプティマイザ比較: $f(x, y) = x^2 + 5y^2$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## まとめ
- 勾配降下法は「下り方向に進む」反復法
- 学習率がすべて: 大きすぎ→発散、小さすぎ→遅い
- ミニバッチで GPU 効率と汎化性能のバランス
- モメンタムで谷を抜ける、Adam で適応的に
- 実践は **Optax (JAX) / torch.optim (PyTorch)** で

## 最適化章 完了 🎉

→ 次の章: [`../../06_ml_math_bridge/README.md`](../../06_ml_math_bridge/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`01_basic_concepts.ipynb`](01_basic_concepts.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../06_ml_math_bridge/README.md`](../../06_ml_math_bridge/README.md) |